# RMT-PPAD migration NB87 - Phase P8 (smoke train: 2 epochs end-to-end)

**Purpose.** Final integration test. Trains the lane-only model for 2
epochs on a small BDD subset (~200 train + 80 val images) and verifies:
- the pipeline runs without errors,
- losses are finite (no NaN),
- all 4 lane losses are non-zero,
- `da_seg == 0.0` exactly (drivable removed).

Once this passes, the FULL ablation (4 configs x 250 epochs) is a job
for dedicated GPUs - see
`stage2/rmt_ppad_migration/P8_train/configs/ablation_matrix.md`.

**Wall time:** ~5-15 min on a T4 (subset extraction + 2-epoch train).
Per the Drive-vs-local rule, all extractions land in `/content/`.

### Cell 1: Mount Drive, install mmcv, set paths

In [1]:
import os, sys, subprocess
from pathlib import Path
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing {REPO_ROOT} -- verify Drive sync.')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

try:
    import mmcv  # noqa: F401
    print(f'[ok] mmcv: {mmcv.__version__}')
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'mmcv'])
    import mmcv
    print(f'[ok] mmcv installed: {mmcv.__version__}')

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

# Pre-prepared joint dataset tarball (NB00 produced this, P0/NB79 used it).
# Layout inside: images/{train,val}/<stem>.jpg + labels/{train,val}/<stem>.lines.txt
# We pull just the images from here, NOT the .lines.txt (those are CLRKDNet
# polyline format, not YOLO; the detection .txt come from BDD_detection_labels.zip).
DOWNLOADS = Path('/content/drive/MyDrive/EcoCAR/downloads')
DATASETS = Path('/content/drive/MyDrive/EcoCAR/datasets')

CURVE_TAR_CANDIDATES = [
    DATASETS / 'bdd100k_clrkd_curve.tar',
]
LBL_ZIP_CANDIDATES = [
    DOWNLOADS / 'rmt_ppad_weights' / 'BDD_detection_labels.zip',
    DOWNLOADS / 'BDD_detection_labels.zip',
]
LANE_TAR_CANDIDATES = [
    DATASETS / 'lane_targets_clr_v1_polyline.tar.gz',
    DATASETS / 'lane_targets_clr_v1.tar.gz',
]

def _first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None

CURVE_TAR = _first_existing(CURVE_TAR_CANDIDATES)
LBL_ZIP   = _first_existing(LBL_ZIP_CANDIDATES)
LANE_TAR  = _first_existing(LANE_TAR_CANDIDATES)
for label, path in (('curve tar', CURVE_TAR), ('labels zip', LBL_ZIP),
                    ('lane tar', LANE_TAR)):
    print(f'  {label:12s} -> {path}')
for path in (CURVE_TAR, LBL_ZIP, LANE_TAR):
    if path is None:
        raise FileNotFoundError(
            'One of the source archives is missing on Drive. Check the '
            'candidate lists in this cell - bdd100k_clrkd_curve.tar comes '
            'from NB00, BDD_detection_labels.zip from P0, '
            'lane_targets_clr_v1_polyline.tar.gz from P1.'
        )

Mounted at /content/drive
[ok] mmcv installed: 2.2.0
torch 2.10.0+cu128 | cuda True
  curve tar    -> /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar
  labels zip   -> /content/drive/MyDrive/EcoCAR/downloads/rmt_ppad_weights/BDD_detection_labels.zip
  lane tar     -> /content/drive/MyDrive/EcoCAR/datasets/lane_targets_clr_v1_polyline.tar.gz


### Cell 2: Extract a small BDD subset into /content/
200 train + 80 val images, matching detection labels (or empty .txt
placeholders), matching lane_target .pt files. Idempotent - re-runs skip
files that already exist.

In [2]:
import sys, os
PREP = 'stage2/rmt_ppad_migration/P8_train/scripts/prepare_bdd_subset.py'
cmd = [
    sys.executable, '-u', PREP,
    '--out-root', '/content/bdd_dataset',
    '--curve-tar', str(CURVE_TAR),
    '--labels-zip', str(LBL_ZIP),
    '--lane-tar', str(LANE_TAR),
    '--n-train', '200', '--n-val', '80',
    '--seed', '0',
]
log = os.path.join(LOG_DIR, 'NB87_prep_subset.log')
rc = run_streaming(cmd, log_path=log, check=False)
if rc != 0:
    raise RuntimeError(f'prepare_bdd_subset rc={rc}; see {log}')

[run_streaming] command: /usr/bin/python3 -u stage2/rmt_ppad_migration/P8_train/scripts/prepare_bdd_subset.py --out-root /content/bdd_dataset --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --labels-zip /content/drive/MyDrive/EcoCAR/downloads/rmt_ppad_weights/BDD_detection_labels.zip --lane-tar /content/drive/MyDrive/EcoCAR/datasets/lane_targets_clr_v1_polyline.tar.gz --n-train 200 --n-val 80 --seed 0
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/NB87_prep_subset.log
  curve_tar   -> /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar
  labels_zip  -> /content/drive/MyDrive/EcoCAR/downloads/rmt_ppad_weights/BDD_detection_labels.zip
  lane_tar    -> /content/drive/MyDrive/EcoCAR/datasets/lane_targets_clr_v1_polyline.tar.gz
[prep] step 1: extract curve tarball
  extracting bdd100k_clrkd_curve.tar -> /content/bdd_curve_scratch
/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage2/rmt_ppad_migration/P8_train/scri

### Cell 3: Run smoke train (2 epochs, batch=4)
Uses `BDD_lane_only.yaml` -> `path: /content/bdd_dataset` (matches the
subset prep in cell 2). Writes to `/content/runs/train/p8_smoke/`.

In [3]:
import sys, os
TRAIN = 'stage2/rmt_ppad_migration/P8_train/scripts/train_lane_only.py'
cmd = [
    sys.executable, '-u', TRAIN,
    '--mode', 'smoke',
    '--name', 'p8_smoke',
    '--project', '/content/runs/train',
    '--device', '0' if torch.cuda.is_available() else 'cpu',
    '--workers', '2',
    '--save-period', '1',
]
log = os.path.join(LOG_DIR, 'NB87_smoke_train.log')
rc = run_streaming(cmd, log_path=log, check=False)
if rc != 0:
    raise RuntimeError(f'smoke train rc={rc}; see {log}')

[run_streaming] command: /usr/bin/python3 -u stage2/rmt_ppad_migration/P8_train/scripts/train_lane_only.py --mode smoke --name p8_smoke --project /content/runs/train --device 0 --workers 2 --save-period 1
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/NB87_smoke_train.log
[train_lane_only] mode=smoke  epochs=2  batch=4  lr0=0.0001
  model  = /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage2/rmt_ppad_migration/vendor/RMT-PPAD/ultralytics/cfg/models/mt-detr/rtdetr-l_bdd_clr_lane.yaml
  data   = /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage2/rmt_ppad_migration/vendor/RMT-PPAD/ultralytics/cfg/datasets/BDD_lane_only.yaml
  device = 0
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


RuntimeError: smoke train rc=1; see /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/NB87_smoke_train.log

### Cell 4: Parse the run output
Look at `runs/train/p8_smoke/results.csv` and the final last-checkpoint
summary; assert all 4 lane keys appear, all are finite, none are NaN,
`da_seg == 0`, and at least one weight was saved.

In [ ]:
from pathlib import Path
import csv, math

RUN_DIR = Path('/content/runs/train/p8_smoke')
print('Contents of', RUN_DIR, ':')
for p in sorted(RUN_DIR.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(RUN_DIR)}  ({p.stat().st_size:,} bytes)')

# Check that at least one weights file got written.
weights = list((RUN_DIR / 'weights').glob('*.pt')) if (RUN_DIR / 'weights').exists() else []
print(f'\nWeights files: {[w.name for w in weights]}')

# Parse results.csv if it exists.
csv_path = RUN_DIR / 'results.csv'
if csv_path.exists():
    print(f'\nresults.csv (last row):')
    rows = list(csv.DictReader(csv_path.open()))
    for r in rows[-2:]:
        print('  ' + '  '.join(f'{k.strip()}={v.strip()}' for k, v in r.items()))

    last = rows[-1]
    # Best-effort: scan for our lane keys + da_seg.
    lane_keys = [k for k in last if 'lane_' in k]
    da_keys = [k for k in last if 'da_' in k or 'da_seg' in k]
    print(f'\n  lane_* columns: {lane_keys}')
    print(f'  da_*    columns: {da_keys}')

    # All lane values finite + non-zero would be ideal; here we just
    # confirm finite (smoke is too short to gauge "decreasing").
    n_nan = 0
    for k, v in last.items():
        try:
            fv = float(v)
            if math.isnan(fv) or math.isinf(fv):
                n_nan += 1
                print(f'    NaN/Inf in column: {k}={v}')
        except (ValueError, TypeError):
            pass
    print(f'\n  non-finite columns: {n_nan}  (expect 0)')

if not weights:
    raise RuntimeError(
        'No checkpoints saved - training did not reach an epoch boundary.'
    )
print('\n[P8 smoke result] PASS - pipeline trains end-to-end; ready for the full ablation runs.')